# Expected Returns Analytics
# 
# This notebook analyzes expected returns using the enhanced v2.0 analytics pipeline:
# - **Monte Carlo Simulation** — Probabilistic upside/downside distributions
# - **Price Target Achievement** — Probability-weighted expected returns by sector
# - **Kalman Filtered Targets** — Noise-reduced price target signals
# - **Analyst Sentiment Features** — Feature-level probability analytics
# - **Cross-Model Comparison** — MC vs Kalman vs Achievement model alignment
#
# Data sources: `analytics.monte_carlo_simulation`, `analytics.price_target_achievement`,
# `analytics.kalman_filtered_price_targets`, `analytics.earnings_probability_analysis`


## 1. Setup & Environment Configuration


In [24]:
import warnings
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

warnings.filterwarnings("ignore")

# Configure database connection
if "DB_URL" not in os.environ:
    env_file = "environment_variables.txt"
    if os.path.exists(env_file):
        with open(env_file) as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    key, value = line.split("=", 1)
                    os.environ[key.strip()] = value.strip()

PLOTLY_TEMPLATE = "plotly_dark"
COLORS = px.colors.qualitative.Dark24

print("✅ Environment configured")


✅ Environment configured


## 2. Data Acquisition


In [49]:
%%sql
SELECT * FROM analytics.monte_carlo_simulation

,ticker,name,sector,industry,region,country,exchange,last_price,pt_median,pt_spread,expected_upside_pct,upside_std,var_5_pct,prob_positive_upside,risk_reward_ratio
0,2223,Saudi Aramco Base Oil Company - Luberef,Materials,Chemicals,Africa / Middle East,SA,SASE,105.20,121.55,15.0,16.084660,2.900628,11.433018,100.00,5.545233
1,2050,Savola Group Company,Consumer Staples,Food Products,Africa / Middle East,SA,SASE,22.88,34.00,13.9,37.286077,13.070576,13.016440,100.00,2.852673
2,4030,The National Shipping Company of Saudi Arabia,Energy,Oil Gas and Consumable Fuels,Africa / Middle East,SA,SASE,31.94,33.00,4.0,3.319631,2.536290,-0.919866,88.92,1.308853
3,KPRMILL,K.P.R. Mill Limited,Consumer Discretionary,Textiles Apparel and Luxury Goods,Asia / Pacific,IN,NSEI,978.85,1092.00,510.0,13.415420,10.663124,-3.989338,88.62,1.258113
4,4613,Kansai Paint Co. Ltd.,Materials,Chemicals,Asia / Pacific,JP,TSE,2611.00,2730.00,900.0,5.002348,7.044622,-6.543204,74.51,0.710095
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5588,INRN,Interroll Holding AG,Industrials,Machinery,Europe,CH,SWX,1938.00,2600.00,691.0,38.481607,7.465267,27.188230,100.00,5.154753
5589,LOTB,Lotus Bakeries NV,Consumer Staples,Food Products,Europe,BE,ENXTBR,10760.00,11000.00,4900.0,-3.698488,9.556410,-21.084774,39.88,-0.387016
5590,KOMN,Komax Holding AG,Industrials,Machinery,Europe,CH,SWX,71.80,68.60,22.0,-4.097962,6.158584,-14.278803,25.99,-0.665407
5591,YSN,secunet Security Networks Aktiengesellschaft,Information Technology,IT Services,Europe,DE,XTRA,203.00,256.00,55.0,18.389196,6.159269,6.783642,100.00,2.985613


In [26]:
%%sql
SELECT * FROM analytics.price_target_achievement WHERE upside_potential NOTNULL AND analyst_conviction NOTNULL

,ticker,name,country,exchange,sector,industry,achievement_probability,upside_potential,price_target_spread_pct,analyst_conviction,eps_revision_momentum,analyst_rating_normalized,expected_return_prob_weighted,confidence_level
0,2223,Saudi Aramco Base Oil Company - Luberef,SA,SASE,Materials,Chemicals,0.75,15.541825,12.340601,100.000000,-0.032690,93.75,11.656369,High
1,2050,Savola Group Company,SA,SASE,Consumer Staples,Food Products,0.20,48.601399,40.882353,33.333333,0.084135,66.75,9.720280,Low
2,MKDM,Mekdam Holding Group - Q.P.S.C.,QA,DSM,Industrials,Industrial Conglomerates,0.70,18.466899,0.000000,100.000000,0.000000,75.00,12.926829,Low
3,MSPW,Musandam Power Company SAOG,OM,MSM,Utilities,Independent Power and Renewable Electricity Pr...,0.81,4.658385,0.000000,0.000000,-0.012820,50.00,3.773292,Low
4,4030,The National Shipping Company of Saudi Arabia,SA,SASE,Energy,Oil Gas and Consumable Fuels,0.81,3.318723,12.121212,50.000000,0.000005,62.50,2.688165,High
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6412,INRN,Interroll Holding AG,CH,SWX,Industrials,Machinery,0.39,34.158927,26.576923,57.142857,0.026040,75.00,13.321981,Medium
6413,LOTB,Lotus Bakeries NV,BE,ENXTBR,Consumer Staples,Food Products,0.79,2.230483,44.545455,40.000000,0.041960,72.50,1.762082,Low
6414,KOMN,Komax Holding AG,CH,SWX,Industrials,Machinery,0.84,-4.456825,32.069971,0.000000,0.000000,55.00,-3.743733,Medium
6415,YSN,secunet Security Networks Aktiengesellschaft,DE,XTRA,Information Technology,IT Services,0.65,26.108374,21.484375,75.000000,0.088565,87.50,16.970443,Medium


In [27]:
%%sql
SELECT * FROM analytics.kalman_filtered_price_targets

,ticker,name,country,exchange,sector,industry,kalman_estimate,kalman_variance,kalman_gain,signal_strength,original_price,original_target,filtered_upside
0,2223,Saudi Aramco Base Oil Company - Luberef,SA,SASE,Materials,Chemicals,120.063650,0.090909,0.909092,10.99999,105.200,121.550,14.128945
1,2050,Savola Group Company,SA,SASE,Consumer Staples,Food Products,32.989100,0.090909,0.909092,10.99999,22.880,34.000,44.183130
2,MKDM,Mekdam Holding Group - Q.P.S.C.,QA,DSM,Industrials,Industrial Conglomerates,3.351819,0.090909,0.909092,10.99999,2.870,3.400,16.788105
3,MSPW,Musandam Power Company SAOG,OM,MSM,Utilities,Independent Power and Renewable Electricity Pr...,0.335636,0.090909,0.909092,10.99999,0.322,0.337,4.234899
4,4030,The National Shipping Company of Saudi Arabia,SA,SASE,Energy,Oil Gas and Consumable Fuels,32.903637,0.090909,0.909092,10.99999,31.940,33.000,3.017023
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6415,INRN,Interroll Holding AG,CH,SWX,Industrials,Machinery,2539.818729,0.090909,0.909092,10.99999,1938.000,2600.000,31.053598
6416,LOTB,Lotus Bakeries NV,BE,ENXTBR,Consumer Staples,Food Products,10978.182017,0.090909,0.909092,10.99999,10760.000,11000.000,2.027714
6417,KOMN,Komax Holding AG,CH,SWX,Industrials,Machinery,68.890906,0.090909,0.909092,10.99999,71.800,68.600,-4.051662
6418,YSN,secunet Security Networks Aktiengesellschaft,DE,XTRA,Information Technology,IT Services,251.181862,0.090909,0.909092,10.99999,203.000,256.000,23.734907


In [28]:
%%sql
SELECT * FROM public.vw_features_analyst_sentiment

,isin,ticker,name,industry,sector,trading_country,region,country,exchange,analyst_bullish_pct,...,pt_median_momentum_1m,pt_median_momentum_3m,pt_acceleration_short,pt_acceleration_long,pt_consensus_convergence,analyst_coverage_change_1m,analyst_coverage_change_3m,analyst_coverage_change_1y,pt_vs_price_momentum,analyst_coverage_trend
0,US67066G1040,NVDA,NVIDIA Corporation,Semiconductors and Semiconductor Equipment,Information Technology,US,United States and Canada,US,NasdaqGS,93.650794,...,0.000000,0.111111,-0.095145,-0.369446,0.263111,1,2,4,0.089512,0.036207
1,US4859241048,KRMN,Karman Holdings Inc.,Aerospace and Defense,Industrials,US,United States and Canada,US,NYSE,80.000000,...,0.452381,0.435294,0.008245,NaN,-0.409354,2,4,9,-0.031792,0.411111
2,US0378331005,AAPL,Apple Inc.,Technology Hardware Storage and Peripherals,Information Technology,US,United States and Canada,US,NasdaqGS,61.702128,...,0.000000,0.083032,-0.026305,-0.116810,-0.014019,0,0,1,0.020300,0.024390
3,US65290E1010,NXT,Nextpower Inc.,Electrical Equipment,Industrials,US,United States and Canada,US,NasdaqGS,78.571429,...,0.173709,0.207729,-0.059961,-0.953108,0.054300,1,1,-2,0.059451,0.046296
4,US02079K3059,GOOGL,Alphabet Inc.,Interactive Media and Services,Communication Services,US,United States and Canada,US,NasdaqGS,87.692308,...,0.166667,0.170213,-0.063828,-0.520520,-0.109644,2,2,7,0.003493,0.040179
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6419,NGCILEASING2,CILEASING,C & I Leasing Plc,Trading Companies and Distributors,Industrials,NG,Africa / Middle East,NG,NGSE,0.000000,...,0.000000,0.000000,0.000000,NaN,0.000000,0,0,1,-0.338158,0.000000
6420,NGBERGER0000,BERGER,Berger Paints Nigeria Plc,Chemicals,Materials,NG,Africa / Middle East,NG,NGSE,NaN,...,0.000000,-0.027947,0.027947,0.000000,0.000000,0,0,0,-0.545079,0.000000
6421,TN0006590012,SIAME,Société Industrielle d'Appareillage et de Maté...,Electrical Equipment,Industrials,TN,Africa / Middle East,TN,BVMT,100.000000,...,0.049869,0.063830,-0.013961,0.223494,0.000000,0,0,0,-0.067739,0.000000
6422,TN0006530018,SOTET,Société Tunisienne d'Entreprises de Télécommun...,Diversified Telecommunication Services,Communication Services,TN,Africa / Middle East,TN,BVMT,100.000000,...,0.000000,0.031206,-0.031206,0.094350,0.000000,0,0,0,-0.176685,0.000000


## 3. Data Overview & Quality Checks


In [29]:
# Rename the DataSpell-imported variables to convenient names
# (Adjust variable names if DataSpell assigns different ones)
try:
    mc = mc_sim.copy()
except NameError:
    print("⚠️ Run the data_input cells above first")

try:
    pt = pt_a.copy()
except NameError:
    print("⚠️ Run the price_target_achievement data_input cell first")

try:
    kal = pt_kal.copy()
except NameError:
    print("⚠️ Run the kalman_filtered_price_targets data_input cell first")

print(f"Monte Carlo Simulation:        {mc.shape[0]:,} stocks × {mc.shape[1]} cols")
print(f"Price Target Achievement:      {pt.shape[0]:,} stocks × {pt.shape[1]} cols")
print(f"Kalman Filtered Targets:       {kal.shape[0]:,} stocks × {kal.shape[1]} cols")

# Summary statistics for core return metrics
display(mc[["expected_upside_pct", "var_5_pct", "prob_positive_upside", "risk_reward_ratio"]].describe().round(2))


Monte Carlo Simulation:        5,593 stocks × 15 cols
Price Target Achievement:      6,417 stocks × 14 cols
Kalman Filtered Targets:       6,420 stocks × 13 cols


,expected_upside_pct,var_5_pct,prob_positive_upside,risk_reward_ratio
count,5593.00,5593.00,5593.00,5593.00
mean,21.71,2.72,74.64,2.73
std,38.96,30.55,32.82,11.92
min,-73.10,-78.51,0.00,-127.79
25%,1.07,-14.81,54.85,0.12
50%,13.39,-1.13,92.95,1.49
75%,31.51,14.50,100.00,3.22
max,584.58,352.09,100.00,538.63


## 4. Monte Carlo Simulation Analysis


### 4.1 Expected Upside Distribution


In [30]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Expected Upside Distribution", "Probability of Positive Return"),
    column_widths=[0.55, 0.45],
)

# Clip extreme outliers for better visualization
upside_clipped = mc["expected_upside_pct"].clip(-100, 300)

fig.add_trace(
    go.Histogram(
        x=upside_clipped,
        nbinsx=80,
        marker_color=COLORS[0],
        opacity=0.75,
        name="Expected Upside %",
    ),
    row=1, col=1,
)
fig.add_vline(x=0, line_dash="dash", line_color="red", row=1, col=1)
fig.add_vline(
    x=mc["expected_upside_pct"].median(),
    line_dash="dot", line_color="green",
    annotation_text=f"Median: {mc['expected_upside_pct'].median():.1f}%",
    row=1, col=1,
)

# Probability of positive return - pie chart
prob_bins = pd.cut(mc["prob_positive_upside"], bins=[0, 25, 50, 75, 100], labels=["0-25%", "25-50%", "50-75%", "75-100%"])
prob_counts = prob_bins.value_counts().sort_index()
fig.add_trace(
    go.Bar(
        x=prob_counts.index.astype(str),
        y=prob_counts.values,
        marker_color=[COLORS[3], COLORS[1], COLORS[0], COLORS[2]],
        name="Stock Count",
    ),
    row=1, col=2,
)

fig.update_layout(
    title="Monte Carlo Simulation: Return Distribution Overview",
    template=PLOTLY_TEMPLATE,
    height=450,
    showlegend=True,
)
fig.update_xaxes(title_text="Expected Upside (%)", row=1, col=1)
fig.update_xaxes(title_text="Probability of Positive Return", row=1, col=2)
fig.update_yaxes(title_text="Number of Stocks", row=1, col=1)
fig.update_yaxes(title_text="Number of Stocks", row=1, col=2)
fig.show()


### 4.2 Risk-Reward by Industry


In [31]:
# Sector-level aggregation
mc_sector = (
    mc.groupby("industry")
    .agg(
        mean_upside=("expected_upside_pct", "mean"),
        median_upside=("expected_upside_pct", "median"),
        mean_var5=("var_5_pct", "mean"),
        mean_prob_positive=("prob_positive_upside", "mean"),
        count=("ticker", "count"),
    )
    .reset_index()
    .sort_values("mean_upside", ascending=False)
)

fig = px.scatter(
    mc_sector,
    x="mean_var5",
    y="mean_upside",
    size="count",
    color="industry",
    hover_name="industry",
    hover_data={"mean_prob_positive": ":.1f", "count": True},
    title="Industry Risk-Reward: Expected Upside vs Value-at-Risk (5%)",
    labels={
        "mean_var5": "Mean VaR 5% (%)",
        "mean_upside": "Mean Expected Upside (%)",
        "count": "# Stocks",
    },
    template=PLOTLY_TEMPLATE,
    height=500,
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.5)
fig.show()


### 4.3 Top Opportunities — Highest Risk-Reward Ratio (Positive Upside)


In [32]:
mc_positive = mc[mc["prob_positive_upside"] >= 75].nlargest(50, "risk_reward_ratio")

fig = px.bar(
    mc_positive,
    x="ticker",
    y="expected_upside_pct",
    color="industry",
    hover_data=["name", "prob_positive_upside", "risk_reward_ratio"],
    title="Top 50 Opportunities: Highest Risk-Reward (≥75% Prob Positive)",
    labels={"expected_upside_pct": "Expected Upside (%)", "ticker": "Ticker"},
    template=PLOTLY_TEMPLATE,
    height=500,
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()


## 5. Price Target Achievement Analysis


### 5.1 Achievement Probability Distribution by Confidence Level


In [33]:
fig = px.violin(
    pt,
    x="confidence_level",
    y="achievement_probability",
    color="confidence_level",
    box=True,
    points="outliers",
    title="Price Target Achievement Probability by Confidence Level",
    labels={
        "achievement_probability": "Achievement Probability",
        "confidence_level": "Confidence Level",
    },
    category_orders={"confidence_level": ["Low", "Medium", "High"]},
    color_discrete_sequence=[COLORS[3], COLORS[1], COLORS[2]],
    template=PLOTLY_TEMPLATE,
    height=450,
)
fig.show()


### 5.2 Probability-Weighted Expected Return by Sector


In [34]:
pt_sector = (
    pt.groupby("industry")
    .agg(
        mean_expected_return=("expected_return_prob_weighted", "mean"),
        median_expected_return=("expected_return_prob_weighted", "median"),
        mean_achievement_prob=("achievement_probability", "mean"),
        mean_conviction=("analyst_conviction", "mean"),
        count=("ticker", "count"),
    )
    .reset_index()
    .sort_values("mean_expected_return", ascending=True)
)

fig = go.Figure()
fig.add_trace(
    go.Bar(
        y=pt_sector["industry"],
        x=pt_sector["mean_expected_return"],
        orientation="h",
        marker_color=[
            COLORS[2] if v >= 0 else COLORS[3]
            for v in pt_sector["mean_expected_return"]
        ],
        text=pt_sector["mean_expected_return"].apply(lambda v: f"{v:.1f}%"),
        textposition="outside",
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Mean Prob-Weighted Return: %{x:.2f}%<br>"
            "Avg Achievement Prob: %{customdata[0]:.0%}<br>"
            "Avg Conviction: %{customdata[1]:.1f}<br>"
            "Stocks: %{customdata[2]}"
        ),
        customdata=pt_sector[["mean_achievement_prob", "mean_conviction", "count"]].values,
    )
)
fig.update_layout(
    title="Probability-Weighted Expected Return by Industry",
    xaxis_title="Mean Expected Return (%)",
    template=PLOTLY_TEMPLATE,
    height=1100,
    margin=dict(l=350),
)
fig.show()


### 5.3 Conviction vs Upside Potential Scatter


In [35]:
fig = px.scatter(
    pt.sample(min(2000, len(pt)), random_state=42),
    x="expected_return_prob_weighted",
    y="upside_potential",
    color="achievement_probability",
    size="analyst_conviction",
    hover_name="ticker",
    hover_data=["name","sector", "industry", "expected_return_prob_weighted","confidence_level"],
    title="Analyst Conviction vs Upside Potential",
    labels={
        "analyst_conviction": "Analyst Conviction (%)",
        "upside_potential": "Upside Potential (%)",
    },
    category_orders={"confidence_level": ["Low", "Medium", "High"]},
    color_discrete_sequence=[COLORS[3], COLORS[1], COLORS[2]],
    template=PLOTLY_TEMPLATE,
    height=500,
    opacity=0.6,
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.4)
fig.show()


## 6. Kalman Filtered Price Target Analysis


### 6.1 Kalman Filtered vs Original Upside


In [36]:
# Pre-compute the column on the full DataFrame
kal["raw_upside"] = (kal["original_target"] - kal["original_price"]) / kal["original_price"] * 100

# Sample AFTER the column exists
kal_sample = kal.sample(min(2000, len(kal)), random_state=42)

fig = px.scatter(
    kal_sample,
    x="filtered_upside",
    y="raw_upside",               # ← now a column name, resolved from the sampled DF
    color="industry",
    hover_name="ticker",
    hover_data=["name", "kalman_estimate", "original_target", "original_price"],
    title="Kalman-Filtered Upside vs Raw Analyst Upside",
    labels={
        "filtered_upside": "Kalman Filtered Upside (%)",
        "raw_upside": "Raw Analyst Upside (%)",
    },
    template=PLOTLY_TEMPLATE,
    height=500,
    opacity=0.6,
)

# Add diagonal reference line
max_range = max(abs(kal_sample["filtered_upside"].quantile(0.01)), abs(kal_sample["filtered_upside"].quantile(0.99)))
fig.add_shape(
    type="line", x0=-max_range, y0=-max_range, x1=max_range, y1=max_range,
    line=dict(color="gray", dash="dash", width=1),
)
fig.show()


### 6.2 Signal Strength Distribution by Sector


In [37]:
fig = px.box(
    kal,
    x="industry",
    y="filtered_upside",
    color="industry",
    title="Kalman-Filtered Upside Distribution by Sector",
    labels={
        "filtered_upside": "Filtered Upside (%)",
        "industry": "",
    },
    template=PLOTLY_TEMPLATE,
    height=1000,
)
fig.update_layout(
    xaxis_tickangle=-65,
    showlegend=False,
)
fig.add_hline(y=0, line_dash="dash", line_color="red", opacity=0.5)
fig.show()


### 6.3 Kalman Noise Reduction Effectiveness


In [38]:
kal["raw_upside"] = (kal["original_target"] - kal["original_price"]) / kal["original_price"] * 100
kal["noise_reduction"] = abs(kal["raw_upside"] - kal["filtered_upside"])

noise_by_sector = (
    kal.groupby("industry")
    .agg(
        mean_noise_reduction=("noise_reduction", "mean"),
        median_raw_upside=("raw_upside", "median"),
        median_filtered_upside=("filtered_upside", "median"),
        count=("ticker", "count"),
    )
    .reset_index()
    .sort_values("mean_noise_reduction", ascending=False)
)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=noise_by_sector["industry"],
    y=noise_by_sector["median_raw_upside"],
    name="Raw Median Upside",
    marker_color=COLORS[1],
    opacity=0.7,
))
fig.add_trace(go.Bar(
    x=noise_by_sector["industry"],
    y=noise_by_sector["median_filtered_upside"],
    name="Kalman-Filtered Median Upside",
    marker_color=COLORS[0],
))
fig.update_layout(
    title="Kalman Filter Impact: Raw vs Filtered Median Upside by Sector",
    yaxis_title="Median Upside (%)",
    barmode="group",
    template=PLOTLY_TEMPLATE,
    height=1000,
    xaxis_tickangle=-85,
)
fig.show()


## 7. Cross-Model Comparison


### 7.1 MC Expected Upside vs Kalman Filtered Upside


In [39]:
# Merge Monte Carlo and Kalman results
mc_kal = mc.merge(
    kal[["ticker","country","exchange", "filtered_upside", "kalman_estimate", "original_price","original_target"]],
    on="ticker",
    how="inner",
)

fig = px.scatter(
    mc_kal.sample(min(2000, len(mc_kal)), random_state=42),
    x="expected_upside_pct",
    y="filtered_upside",
    color="industry",
    hover_name="ticker",
    hover_data=["name", "original_price","kalman_estimate","original_target","prob_positive_upside"],
    title="Monte Carlo vs Kalman-Filtered Expected Returns",
    labels={
        "expected_upside_pct": "MC Expected Upside (%)",
        "filtered_upside": "Kalman Filtered Upside (%)",
    },
    template=PLOTLY_TEMPLATE,
    height=500,
    opacity=0.5,
)
# Diagonal reference
fig.add_shape(
    type="line", x0=-100, y0=-100, x1=200, y1=200,
    line=dict(color="gray", dash="dash", width=1),
)
fig.show()

In [40]:

# Correlation summary
corr = mc_kal[["expected_upside_pct", "filtered_upside"]].corr().iloc[0, 1]
print(f"📊 MC ↔ Kalman correlation: {corr:.3f}")


📊 MC ↔ Kalman correlation: 0.923


### 7.2 Tri-Model Alignment: MC + Kalman + Achievement


In [41]:
# Merge all three models
tri = (
    mc[["ticker", "name", "sector","industry", "expected_upside_pct", "prob_positive_upside"]]
    .merge(
        kal[["ticker", "filtered_upside"]],
        on="ticker",
        how="inner",
    )
    .merge(
        pt[["ticker", "expected_return_prob_weighted", "achievement_probability", "confidence_level"]],
        on="ticker",
        how="inner",
    )
)

# Agreement score: all three models agree on direction
tri["mc_bullish"] = tri["expected_upside_pct"] > 0
tri["kal_bullish"] = tri["filtered_upside"] > 0
tri["pt_bullish"] = tri["expected_return_prob_weighted"] > 0
tri["agreement_score"] = (
    tri["mc_bullish"].astype(int)
    + tri["kal_bullish"].astype(int)
    + tri["pt_bullish"].astype(int)
)
tri["signal"] = tri["agreement_score"].map(
    {0: "Strong Bearish (0/3)", 1: "Bearish (1/3)", 2: "Bullish (2/3)", 3: "Strong Bullish (3/3)"}
)

fig = px.histogram(
    tri,
    x="signal",
    color="signal",
    title="Tri-Model Signal Agreement (MC + Kalman + Achievement)",
    labels={"signal": "Model Agreement", "count": "Number of Stocks"},
    color_discrete_map={
        "Strong Bearish (0/3)": COLORS[3],
        "Bearish (1/3)": COLORS[1],
        "Bullish (2/3)": COLORS[0],
        "Strong Bullish (3/3)": COLORS[2],
    },
    category_orders={"signal": [
        "Strong Bearish (0/3)", "Bearish (1/3)",
        "Bullish (2/3)", "Strong Bullish (3/3)",
    ]},
    template=PLOTLY_TEMPLATE,
    height=420,
)
fig.update_layout(showlegend=False)
fig.show()

In [42]:

print(f"\n📊 Model Agreement Summary:")
print(tri["signal"].value_counts().to_string())



📊 Model Agreement Summary:
signal
Strong Bullish (3/3)    4108
Strong Bearish (0/3)    1027
Bullish (2/3)            262
Bearish (1/3)            196


### 7.3 Strong Consensus Picks — All 3 Models Bullish, High Confidence


In [43]:
strong_consensus = (
    tri[
        (tri["agreement_score"] == 3)
        & (tri["prob_positive_upside"] >= 55)
        & (tri["achievement_probability"] >= 0.6)
    ]
    .nlargest(50, "expected_upside_pct")
)

if len(strong_consensus) > 0:
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=strong_consensus["ticker"],
        y=strong_consensus["expected_upside_pct"],
        name="MC Expected Upside",
        marker_color=COLORS[0],
    ))
    fig.add_trace(go.Bar(
        x=strong_consensus["ticker"],
        y=strong_consensus["filtered_upside"],
        name="Kalman Filtered Upside",
        marker_color=COLORS[1],
    ))
    fig.add_trace(go.Bar(
        x=strong_consensus["ticker"],
        y=strong_consensus["expected_return_prob_weighted"],
        name="Prob-Weighted Return",
        marker_color=COLORS[2],
    ))
    fig.update_layout(
        title=f"Top {len(strong_consensus)} Strong Consensus Picks (All 3 Models Bullish)",
        yaxis_title="Expected Return (%)",
        barmode="group",
        template=PLOTLY_TEMPLATE,
        height=500,
        xaxis_tickangle=-45,
    )
    fig.show()

    display(
        strong_consensus[["ticker", "name", "sector","industry", "expected_upside_pct",
                          "filtered_upside", "expected_return_prob_weighted",
                          "prob_positive_upside", "achievement_probability", "confidence_level"]]
        .reset_index(drop=True)
    )
else:
    print("No stocks meet the strong consensus criteria.")


,ticker,name,sector,industry,expected_upside_pct,filtered_upside,expected_return_prob_weighted,prob_positive_upside,achievement_probability,confidence_level
0,TLEVISACPO,Grupo Televisa S.A.B.,Communication Services,Diversified Telecommunication Services,131.309440,8.174394,6.024523,98.39,0.67,Low
1,BHVN,Biohaven Ltd.,Health Care,Biotechnology,94.713776,22.905128,15.369327,95.34,0.61,Low
2,ARA,Consorcio ARA S. A. B. de C. V.,Consumer Discretionary,Household Durables,78.597596,2.272729,1.550000,98.54,0.62,Low
3,SFOR,S4 Capital plc,Communication Services,Media,71.009426,5.369381,4.488798,93.92,0.76,Low
4,BHIA3,Grupo Casas Bahia S.A.,Consumer Discretionary,Specialty Retail,66.151784,5.603990,3.821918,95.39,0.62,Low
5,PLUG,Plug Power Inc.,Industrials,Electrical Equipment,57.672932,6.493512,4.428571,80.26,0.62,Low
6,CPI,Capita plc,Industrials,Professional Services,52.758188,21.555783,14.226804,100.00,0.60,Low
7,ORBIA,Orbia Advance Corporation S.A.B. de C.V.,Materials,Chemicals,43.714761,4.929636,3.633138,81.75,0.67,Low
8,LNTH,Lantheus Holdings Inc.,Health Care,Health Care Equipment and Supplies,41.422706,16.436423,11.571231,100.00,0.64,Low
9,GPCR,Structure Therapeutics Inc.,Health Care,Pharmaceuticals,40.172044,26.571324,17.537058,100.00,0.60,Low


## 8. Sector Expected Returns Heatmap


In [44]:
# Aggregate all return metrics by sector
sector_returns = (
    tri.groupby("industry")
    .agg(
        mc_mean=("expected_upside_pct", "mean"),
        mc_median=("expected_upside_pct", "median"),
        kalman_mean=("filtered_upside", "mean"),
        kalman_median=("filtered_upside", "median"),
        pt_mean=("expected_return_prob_weighted", "mean"),
        pt_median=("expected_return_prob_weighted", "median"),
        pct_bullish=("agreement_score", lambda x: (x == 3).mean() * 100),
        count=("ticker", "count"),
    )
    .reset_index()
)

heatmap_data = sector_returns.set_index("industry")[
    ["mc_mean", "mc_median", "kalman_mean", "kalman_median", "pt_mean", "pt_median", "pct_bullish"]
].rename(columns={
    "mc_mean": "MC Mean",
    "mc_median": "MC Median",
    "kalman_mean": "Kalman Mean",
    "kalman_median": "Kalman Median",
    "pt_mean": "Achiev. Mean",
    "pt_median": "Achiev. Median",
    "pct_bullish": "% All Bullish",
})

fig = px.imshow(
    heatmap_data.round(1),
    color_continuous_scale="RdYlGn",
    text_auto=True,
    aspect="auto",
    title="Sector Expected Returns Heatmap (All Models)",
    labels={"color": "Value"},
)
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    height=1250,
)
fig.show()


## 9. VaR & Tail Risk Analysis


In [45]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("VaR 5% Distribution", "VaR 5% vs Expected Upside"),
)

# VaR distribution
var_clipped = mc["var_5_pct"].clip(-150, 300)
fig.add_trace(
    go.Histogram(
        x=var_clipped,
        nbinsx=80,
        marker_color=COLORS[3],
        opacity=0.75,
        name="VaR 5%",
    ),
    row=1, col=1,
)
fig.add_vline(x=0, line_dash="dash", line_color="blue", row=1, col=1)

# VaR vs Expected Upside (sampled for performance)
sample = mc.sample(min(2000, len(mc)), random_state=42)
fig.add_trace(
    go.Scatter(
        x=sample["var_5_pct"],
        y=sample["expected_upside_pct"],
        mode="markers",
        marker=dict(
            size=4,
            color=sample["prob_positive_upside"],
            colorscale="RdYlGn",
            colorbar=dict(title="P(+)"),
            opacity=0.5,
        ),
        name="Stocks",
    ),
    row=1, col=2,
)
fig.add_shape(
    type="line", x0=-100, y0=-100, x1=300, y1=300,
    line=dict(color="gray", dash="dash", width=1),
    row=1, col=2,
)

fig.update_layout(
    title="Value-at-Risk (5%) Analysis",
    template=PLOTLY_TEMPLATE,
    height=450,
    showlegend=False,
)
fig.update_xaxes(title_text="VaR 5% (%)", row=1, col=1)
fig.update_xaxes(title_text="VaR 5% (%)", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Expected Upside (%)", row=1, col=2)
fig.show()


## 10. Summary Statistics


In [46]:
summary = {
    "Monte Carlo": {
        "Stocks Analyzed": len(mc),
        "Mean Expected Upside (%)": mc["expected_upside_pct"].mean().round(2),
        "Median Expected Upside (%)": mc["expected_upside_pct"].median().round(2),
        "% Stocks with Positive Upside": (mc["expected_upside_pct"] > 0).mean() * 100,
        "Mean Prob Positive (%)": mc["prob_positive_upside"].mean().round(1),
    },
    "Price Target Achievement": {
        "Stocks Analyzed": len(pt),
        "Mean Prob-Weighted Return (%)": pt["expected_return_prob_weighted"].mean().round(2),
        "Mean Achievement Prob": pt["achievement_probability"].mean().round(3),
        "High Confidence Count": (pt["confidence_level"] == "High").sum(),
        "Mean Analyst Conviction (%)": pt["analyst_conviction"].mean().round(1),
    },
    "Kalman Filter": {
        "Stocks Analyzed": len(kal),
        "Mean Filtered Upside (%)": kal["filtered_upside"].mean().round(2),
        "Median Filtered Upside (%)": kal["filtered_upside"].median().round(2),
        "Mean Signal Strength": kal["signal_strength"].mean().round(2),
        "% Positive Filtered Upside": (kal["filtered_upside"] > 0).mean() * 100,
    },
}

summary_df = pd.DataFrame(summary).T
display(summary_df)

if len(tri) > 0:
    print(f"\n🔗 Cross-Model Coverage: {len(tri):,} stocks in all 3 models")
    print(f"   Strong Bullish (3/3 agree): {(tri['agreement_score'] == 3).sum():,} ({(tri['agreement_score'] == 3).mean()*100:.1f}%)")
    print(f"   Strong Bearish (0/3 agree): {(tri['agreement_score'] == 0).sum():,} ({(tri['agreement_score'] == 0).mean()*100:.1f}%)")
    print(f"   MC ↔ Kalman correlation:    {tri[['expected_upside_pct', 'filtered_upside']].corr().iloc[0,1]:.3f}")
    print(f"   MC ↔ Achievement corr:      {tri[['expected_upside_pct', 'expected_return_prob_weighted']].corr().iloc[0,1]:.3f}")

print("\n✅ Expected Returns Analytics complete")


,Stocks Analyzed,Mean Expected Upside (%),Median Expected Upside (%),% Stocks with Positive Upside,Mean Prob Positive (%),Mean Prob-Weighted Return (%),Mean Achievement Prob,High Confidence Count,Mean Analyst Conviction (%),Mean Filtered Upside (%),Median Filtered Upside (%),Mean Signal Strength,% Positive Filtered Upside
Monte Carlo,5593.0,21.71,13.39,76.953335,74.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Price Target Achievement,6417.0,NaN,NaN,NaN,NaN,8.3,0.614,1031.0,62.2,NaN,NaN,NaN,NaN
Kalman Filter,6420.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.54,12.23,11.0,77.258567



🔗 Cross-Model Coverage: 5,593 stocks in all 3 models
   Strong Bullish (3/3 agree): 4,108 (73.4%)
   Strong Bearish (0/3 agree): 1,027 (18.4%)
   MC ↔ Kalman correlation:    0.923
   MC ↔ Achievement corr:      0.743

✅ Expected Returns Analytics complete
